# NB4｜應用一：奶粉三聚氰胺摻偽篩檢

**食品分析｜拉曼光譜與 RamanSPy 入門系列（第 4 本，共 5 本）**

2008 年中國毒奶粉事件中，不肖業者加入三聚氰胺以虛增蛋白質檢驗值（凱氏定氮法只測總氮）。拉曼光譜能不能當快篩工具？這一本用 60 個樣品實際檢驗。

---
### 這一本你會學到
- 用診斷峰 + 3SD 法則建立判定閾值
- 求出實務偵測極限（LOD）並誠實面對它的限制
- 用 PCA 做無監督的整體樣貌檢視
- 用 KMeans 分群，並解釋為什麼它「分錯」

> 💡 **完全沒寫過程式也沒關係。** 你只要做三件事：
> 1. 用滑鼠點每一格左邊的 ▶ 播放鍵（或按 `Shift + Enter`）
> 2. 看下面跑出來的圖和數字
> 3. 遇到 `# 👉 換你做` 的地方，照提示改一個數字或一個字，再跑一次


In [ ]:
# ===== 第一次執行請先跑這一格（大約 1 分鐘）=====
# 在 Google Colab 上，套件不是永久安裝的，每次重開都要跑一次。
!pip install -q ramanspy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ramanspy as rp

# 讓圖上的中文正常顯示（Colab 用）
!wget -q -O TaipeiSans.ttf https://drive.google.com/uc?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_ 2>/dev/null
import matplotlib
try:
    matplotlib.font_manager.fontManager.addfont("TaipeiSans.ttf")
    matplotlib.rc("font", family="Taipei Sans TC Beta")
except Exception:
    pass
matplotlib.rcParams["axes.unicode_minus"] = False

print("準備完成！")


In [ ]:
# ===== 資料載入設定 =====
# 這一行由老師部署時自動填入正確的 GitHub 網址，學生不用改。
DATA_BASE = "https://raw.githubusercontent.com/Tai-ShengYeh/Tai-ShengYeh.github.io/main/ramanspy-food-analysis/data/"

# 若你把 CSV 直接上傳到 Colab 左側「檔案」，把上面那行改成： DATA_BASE = ""
# 若你在自己電腦跑，且 data 資料夾就在旁邊，改成：       DATA_BASE = "data/"

def load_spectra(filename):
    """讀 CSV → 回傳 (樣品資訊表 meta, ramanspy 光譜物件 spectra)"""
    df = pd.read_csv(DATA_BASE + filename)
    meta_cols = [c for c in df.columns if not c.replace(".", "", 1).isdigit()]
    axis = np.array([float(c) for c in df.columns if c not in meta_cols])
    spectra = rp.SpectralContainer(df.drop(columns=meta_cols).values, axis)
    return df[meta_cols].reset_index(drop=True), spectra

print("load_spectra() 已定義，資料來源：", DATA_BASE or "（Colab 本機檔案）")


In [ ]:
# 本課程統一使用的標準前處理流程
pipeline = rp.preprocessing.Pipeline([
    rp.preprocessing.misc.Cropper(region=(450, 1800)),          # 裁切
    rp.preprocessing.despike.WhitakerHayes(),                    # 去宇宙射線
    rp.preprocessing.denoise.SavGol(window_length=9, polyorder=3),  # 平滑
    rp.preprocessing.baseline.IModPoly(),                        # 基線校正
    rp.preprocessing.normalise.MinMax(),                         # 歸一化
])


## 1. 載入資料

60 個奶粉樣品：20 個正常、40 個摻入 0.2–5.0% 三聚氰胺。

In [ ]:
meta, spectra = load_spectra("milk_powder_melamine.csv")
processed = pipeline.apply(spectra)

print(meta.label.value_counts())
meta.head()

In [ ]:
X = processed.spectral_data
axis = processed.spectral_axis

plt.figure(figsize=(11, 3.5))
for pct in [0, 0.5, 1.5, 3.0, 5.0]:
    i = int((meta.melamine_pct - pct).abs().idxmin())
    plt.plot(axis, X[i], lw=.9, label=f"{meta.melamine_pct[i]:.1f}%")
plt.axvline(676, ls=":", color="r")
plt.legend(title="三聚氰胺"); plt.xlabel("拉曼位移 (cm$^{-1}$)")
plt.title("摻偽濃度越高，676 cm$^{-1}$ 越明顯")
plt.show()

## 2. 診斷峰法：抓 676 cm⁻¹

三聚氰胺的三嗪環呼吸振動在 676 cm⁻¹，而奶粉基質（乳糖、酪蛋白）在這裡剛好沒有峰 —— 這叫**乾淨的分析視窗**，是好的診斷峰的必要條件。

In [ ]:
i676 = np.argmin(abs(axis - 676))
peak676 = X[:, i676]

plt.figure(figsize=(7, 4))
plt.scatter(meta.melamine_pct, peak676, s=30)
plt.xlabel("加入的三聚氰胺 (%)"); plt.ylabel("676 cm$^{-1}$ 強度")
plt.title("劑量–反應關係")
plt.show()

## 3. 判定閾值與偵測極限（LOD）

分析化學的標準做法：**閾值 = 空白樣品的平均值 + 3 × 標準差**。

超過這條線就判定「檢出」，落在線下就是「未檢出」。

In [ ]:
blank = peak676[meta.label == "normal"]
threshold = blank.mean() + 3 * blank.std()
print(f"空白樣品 676 強度：平均 {blank.mean():.4f}，標準差 {blank.std():.4f}")
print(f"判定閾值 = {threshold:.4f}")

detected = peak676 > threshold
lod = meta.melamine_pct[detected & (meta.melamine_pct > 0)].min()
missed = sorted(meta.melamine_pct[(~detected) & (meta.melamine_pct > 0)].values)

print(f"\n實務偵測極限 LOD ≈ {lod:.2f} %")
print(f"漏檢的濃度（偽陰性）：{missed}")
print(f"偽陽性（正常品被誤判）：{int((detected & (meta.label=='normal')).sum())} 個")

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(meta.melamine_pct, peak676, s=32, c=np.where(detected, "crimson", "grey"))
plt.axhline(threshold, ls="--", color="crimson")
plt.text(3.2, threshold * 1.05, "判定閾值（空白 + 3SD）", color="crimson")
plt.axvline(lod, ls=":", color="green")
plt.text(lod + 0.1, peak676.max() * 0.5, f"LOD ≈ {lod:.1f}%", color="green")
plt.xlabel("加入的三聚氰胺 (%)"); plt.ylabel("676 cm$^{-1}$ 強度")
plt.title("紅點 = 判定檢出")
plt.show()

## 4. ⚠️ 最重要的一課：這個方法夠用嗎？

我們算出 LOD ≈ 0.5%，也就是 **5,000 mg/kg**。

而國際食品法典（Codex, CXS 193-1995）對三聚氰胺的限量是：

| 品項 | 限量 |
|---|---|
| 粉狀嬰兒配方 | 1 mg/kg |
| 液態嬰兒配方 | 0.15 mg/kg |
| 其他食品與飼料 | 2.5 mg/kg |

**差距大約 2,000 倍以上。**

所以正確的結論是：
- ✅ 一般拉曼光譜可以當作**現場快篩**，抓出「明目張膽的大量摻假」
- ❌ 它**不能**取代 LC-MS/MS 這類法規確認方法
- 💡 要逼近法規限值，需要 **SERS（表面增強拉曼）**；文獻報導可將牛奶中三聚氰胺的 LOD 壓到 0.02–1 mg/L

> **「未檢出」不等於「沒有」。** 任何檢驗報告都必須同時標示方法的偵測極限，否則這句話沒有意義。

## 5. PCA：不告訴電腦答案，它能自己看出什麼？

PCA（主成分分析）把 676 個波數壓縮成 2–3 個新座標，是探索資料的第一步。

⚠️ **RamanSPy 的回傳格式要注意**：`projections` 是一個 list，`projections[0]` 是所有樣品的 PC1 分數。

In [ ]:
pca = rp.analysis.decompose.PCA(n_components=3)
projections, components = pca.apply(processed)

pc1, pc2, pc3 = projections      # 每個都是長度 60 的陣列

plt.figure(figsize=(6.5, 4.5))
sc = plt.scatter(pc1, pc2, c=meta.melamine_pct, cmap="plasma", s=45, edgecolor="w")
plt.colorbar(sc, label="三聚氰胺 (%)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("PCA 分數圖：樣品自動排成一條濃度軸")
plt.show()

print("PC1 與濃度的相關係數 =", round(np.corrcoef(pc1, meta.melamine_pct)[0, 1], 3))

### 一定要看負荷量（loadings）！

分數圖分得開，不代表模型是對的 —— 有可能它抓到的是「量測那天溼度不同」之類的假訊號。
**看負荷量，確認模型盯的是化學上說得通的峰。**

In [ ]:
plt.figure(figsize=(9, 3.2))
plt.plot(axis, components[0], lw=1)
plt.axvline(676, ls=":", color="r"); plt.text(690, components[0].max()*0.8, "676 cm$^{-1}$", color="r")
plt.xlabel("拉曼位移 (cm$^{-1}$)"); plt.title("PC1 負荷量：模型主要在看 676 cm$^{-1}$ ✔")
plt.show()

## 6. KMeans 分群：以及它為什麼「分錯」

In [ ]:
kmeans = rp.analysis.cluster.KMeans(n_clusters=2)
distances, centers = kmeans.apply(processed)

# distances[0]、distances[1] 是到兩個群中心的距離 → 取較近的那一群
labels = np.argmin(np.array(distances), axis=0)

print(pd.crosstab(labels, meta.label, rownames=["分群結果"], colnames=["真實標籤"]))
print()
for g in [0, 1]:
    print(f"第 {g} 群的三聚氰胺濃度範圍：{meta.melamine_pct[labels==g].min():.2f} – {meta.melamine_pct[labels==g].max():.2f} %")

### 為什麼分群結果和標籤對不起來？

因為 KMeans 是**無監督**的 —— 它不知道「正常 / 摻偽」這個定義，它只會照**光譜的相似度**分堆。

而摻了 0.2% 的樣品，光譜長得幾乎和正常品一模一樣（低於 LOD），所以被歸到同一群，**這在光譜上是正確的**。

**結論**：分群結果對不上標籤，通常不是演算法壞掉，而是在告訴你 **「這兩類在你量的訊號上本來就分不開」**。這是一個非常有價值的資訊。

### 🧪 自我檢核

1. 為什麼選 676 cm⁻¹ 當診斷峰，而不是三聚氰胺其他的峰？
2. 閾值用「空白 + 3SD」，這代表偽陽性率大約多少？
3. 檢驗報告寫「三聚氰胺未檢出」，一個食品分析師應該追問什麼？
4. PCA 分數圖把兩組分得很開，可以直接發表說「拉曼能區分摻偽奶粉」嗎？

<details><summary>▶ 點開看參考答案</summary>

1. 因為 676 cm⁻¹ 同時滿足兩個條件：(a) 它是三聚氰胺最強的峰；(b) 奶粉基質在該處沒有干擾峰（乾淨的分析視窗）。診斷峰的價值來自「強」+「不重疊」。
2. 常態分佈下超過 +3SD 的機率約 0.13%，所以偽陽性率約 0.1%。（3SD 對應 LOD；定量極限 LOQ 通常用 10SD。）
3. 追問**方法的偵測極限是多少**，以及是否低於法規限量。用 LOD = 0.5% 的方法測出「未檢出」，對 2.5 mg/kg 的法規限量完全沒有意義。
4. 不行。必須 (a) 檢查負荷量確認抓到的是化學訊號、(b) 用獨立的驗證集或交叉驗證，而不是只看訓練資料的分數圖。分數圖分得開很容易，過度配適也很容易。

</details>


---
### 📚 這一本用到的資料與文獻

**資料**：`data/` 內的光譜為**依文獻峰位建立的模擬資料**（`make_data.py`，亂數種子 20260801），刻意加入螢光背景、宇宙射線與雜訊。可用於教學演練，**不可引用為實驗證據**。

**主要文獻**

- Georgiev, D. et al. *RamanSPy: An Open-Source Python Package for Integrative Raman Spectroscopy Data Analysis*. **Anal. Chem.** 2024, 96(21), 8492–8500. doi:10.1021/acs.analchem.4c00383
- Gill, D.; Kilponen, R. G.; Rimai, L. *Resonance Raman Scattering … in Intact Plant Tissues*. **Nature** 1970, 227, 743–744. doi:10.1038/227743a0
- Lu, L. et al. *Resonance Raman scattering of β-carotene … second singlet state*. **J. Photochem. Photobiol. B** 2018, 179, 18–22. doi:10.1016/j.jphotobiol.2017.12.022
- Withnall, R. et al. *Raman spectra of carotenoids in natural products*. **Spectrochim. Acta A** 2003, 59(10), 2207–2212. doi:10.1016/S1386-1425(03)00064-7
- de Oliveira, V. E. et al. *Carotenes and carotenoids in natural biological samples*. **J. Raman Spectrosc.** 2010, 41(6), 642–650. doi:10.1002/jrs.2493
- Portarena, S. et al. *Cultivar discrimination, fatty acid profile and carotenoid characterization of monovarietal olive oils by Raman spectroscopy at a single glance*. **Food Control** 2019, 96, 137–145. doi:10.1016/j.foodcont.2018.09.011
- Chen, Y. et al. *Quantitative analysis of β-carotene and unsaturated fatty acids in blended olive oil via Raman spectroscopy combined with model prediction*. **Food Chemistry** 2025, 470, 142621. doi:10.1016/j.foodchem.2024.142621
- Schmidt, W. et al. *Continuous Temperature-Dependent Raman Spectroscopy of Melamine and Structural Analog Detection in Milk Powder*. **Appl. Spectrosc.** 2015, 69(3), 398–406. doi:10.1366/14-07600
- Zhang, X. et al. *Detection of melamine in liquid milk using SERS*. **J. Raman Spectrosc.** 2010, 41(12), 1655–1660. doi:10.1002/jrs.2629
- Kim, A. et al. *Melamine Sensing in Milk Products by Using SERS*. **Anal. Chem.** 2012, 84(21), 9303–9309. doi:10.1021/ac302025q
- FAO/WHO Codex Alimentarius. *General Standard for Contaminants and Toxins in Food and Feed*, **CXS 193-1995**.

完整清單見課程網站的「數據來源」與「參考文獻」兩節。